# ICS5111 · Session 02: practice notebook
## Practice and Further Reading

This notebook repeats the main steps of Session 02 with functions you write yourself. Work in pairs or alone; it takes about an hour and a half. The Practice and Further Reading sheet on the VLE lists what to keep.

| Exercise | What you do | Walkthrough section |
|---|---|---|
| R0 | write the context-window function | W2 |
| R1 | write one node2vec step and a walk sampler | W5 |
| **R2 (priority)** | choose $p$ and $q$ on validation data, test once, measure seed variation | W7 |
| R3 | ranking metrics by hand; four edge operators for link prediction | W8 |
| **R4 (priority)** | find and fix two leaks in a link-prediction pipeline | W8 |
| R5 | a two-hop neighbour average for new papers | W9 |
| R6 (optional) | TransE with a one-to-many relation | W10 |
| R7 | reading note | |

**How the checks work.** Write your code where it says `TODO` and run the check below it; a check prints whether your result matches, with a hint when it does not. A blank TODO stops with a message rather than an error, so you can run the whole notebook at any time. The checks confirm numbers, not interpretations: the written answers are yours. Keep the fixed Cora split and reference configuration; any setting you change is judged on **validation** accuracy, and the test set is used once, at the end of an exercise.

### Setup

The next two cells install the course library, `ai-learning-kit`, and start Session 02: the helper functions (`T`) that your own functions are compared with, and Cora, which the library downloads from its publisher, LINQS, the first time it is needed. In Notebook Tutor the first cell is skipped, because the tutor already includes the library; in Colab the install takes about a minute (choose *Run anyway* if Colab warns that the notebook was not written by Google). Nothing has to be downloaded or uploaded by hand.

In [ ]:
%pip install -q "ai-learning-kit[graphs]@https://github.com/charlieabela/ai-learning-kit/archive/refs/tags/v0.5.0.zip"

In [2]:
import ai_learning_kit as kit
kit.start("ICS5111", "S02")       # also downloads Cora from LINQS the first time
T = kit.tools()                   # the Session 02 helper functions

import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt
from IPython.display import display
import json, platform
SEED = T.SEED
print("helpers", T.VERSION, "| Python", platform.python_version(), "| numpy", np.__version__, "| networkx", nx.__version__)

ai-learning-kit 0.5.0 · ICS5111 S02: Graph embeddings
Data: Cora citation network (LINQS), from LINQS Statistical Relational Learning Group, UC Santa Cruz (checked)
Helpers: T = kit.tools(). This session has no answer book: the notebook checks itself.
helpers 2026-10-27 | Python 3.12.3 | numpy 2.5.4 | networkx 3.7


In [3]:
toy = T.toy_graph()
G, X, y, paper_ids, classes, split = T.load_cora()
REFERENCE = dict(n_walks=3, length=16, window=2, dim=16, negatives=3, epochs=2, lr=.03)   # the W7 configuration
print("seven-node graph and Cora loaded:", toy.number_of_nodes(), "and", G.number_of_nodes(), "nodes")

seven-node graph and Cora loaded: 7 and 2708 nodes


## R0 · The context-window function
Complete `my_context_pairs(walk, window)`. It returns the **ordered** (centre, context) pairs of one walk: every position is paired with the positions up to `window` before and after it, excluding the centre **position** (not every repeated node). Repeated observations are kept. The check compares your function with the helper on three walks, including one that revisits nodes.

Before running, predict the number of pairs for 1, 2, 4, 5, 6 with window 2 and with window 1, and explain why a context can be two positions away without being adjacent in the graph.

In [4]:
def my_context_pairs(walk, window):
    pairs = []
    for i, centre in enumerate(walk):
        # TODO: set the first and last+1 positions of the window around position i.
        start, stop = None, None
        if start is None or stop is None:
            return None
        for j in range(start, stop):
            if j != i:
                pairs.append((centre, walk[j]))
    return pairs

In [5]:
# Check R0
tests = [([1, 2, 4, 5, 6], 2), ([1, 2, 4, 5, 6], 1), ([1, 3, 2, 3, 1, 2], 2)]
if my_context_pairs([1, 2], 1) is None:
    print("R0: complete start and stop, then rerun. Hints: max(0, ...) and min(len(walk), ...).")
else:
    for walk, window in tests:
        mine = my_context_pairs(walk, window)
        ref = [tuple(map(int, p)) for p in T.context_pairs([walk], window)]
        print(f"walk {walk}, window {window}: {len(mine)} pairs -", "matches the helper" if mine == ref else f"differs (expected {len(ref)} pairs: {ref})")

R0: complete start and stop, then rerun. Hints: max(0, ...) and min(len(walk), ...).


**My answer:** a context two positions away …; the repeated walk produces the pair (3, 3) because …

## R1 · One node2vec step, and a walk
Complete `node2vec_step(g, previous, current, p, q, rng)`. It returns the next node: each neighbour $x$ of `current` gets weight $1/p$ if it is `previous`, $1$ if it is also a neighbour of `previous`, and $1/q$ otherwise; the weights are normalised and one neighbour is drawn with `rng.choice`. On the first step of a walk there is no previous node, and every neighbour gets weight 1. Then `my_walk` uses your step to sample a whole walk.

The check draws the next step 5,000 times from the state (previous 1, current 2) with $p = 2$, $q = 0.5$, and compares the frequencies with 1/7, 2/7 and 4/7 (tolerance 0.02).

In [6]:
def node2vec_step(g, previous, current, p, q, rng):
    candidates = sorted(g.neighbors(current))
    weights = []
    for x in candidates:
        # TODO: weight 1 if previous is None; 1/p for a return; 1 if x is a neighbour of previous; 1/q otherwise.
        w = None
        if w is None:
            return None
        weights.append(w)
    weights = np.asarray(weights, dtype=float)
    # TODO: normalise the weights and draw one candidate with rng.choice(len(candidates), p=...).
    return None

def my_walk(g, start, length, p, q, rng):
    walk = [start]
    while len(walk) < length:
        previous = walk[-2] if len(walk) > 1 else None
        nxt = node2vec_step(g, previous, walk[-1], p, q, rng)
        if nxt is None:
            return None
        walk.append(nxt)
    return walk

In [7]:
# Check R1
rng = np.random.default_rng(SEED)
if node2vec_step(toy, 1, 2, 2, .5, rng) is None:
    print("R1: complete the weight and the draw, then rerun. Walkthrough W5 gives the expected probabilities.")
else:
    draws = np.array([node2vec_step(toy, 1, 2, 2, .5, rng) for _ in range(5000)])
    freq = {x: float(np.mean(draws == x)) for x in [1, 3, 4]}
    theory = {1: 1/7, 3: 2/7, 4: 4/7}
    ok = all(abs(freq[x] - theory[x]) < .02 for x in theory)
    print("frequencies from 5,000 draws:", {x: round(v, 3) for x, v in freq.items()}, "| theory:", {x: round(v, 3) for x, v in theory.items()})
    print("R1 step:", "matches within 0.02" if ok else "differs: check the distance from the previous node and the normalisation")
    walk = my_walk(toy, 1, 12, 1, .25, rng)
    adjacent = all(toy.has_edge(a, b) for a, b in zip(walk, walk[1:]))
    print("a walk with p=1, q=0.25:", walk, "|", "every step follows an edge" if adjacent and len(walk) == 12 else "check my_walk")

R1: complete the weight and the draw, then rerun. Walkthrough W5 gives the expected probabilities.


**My answer:** with $q = 0.25$ the walk tends to … because …

## R2 (priority) · Choose $p$ and $q$ on validation data, then test once
The next cell embeds Cora for nine settings of $p$ and $q$, everything else as in W7, and records the **validation** accuracy of each (about half a minute). Then:

1. complete `chosen` with the setting you select from the validation table, and justify the choice in a sentence;
2. evaluate that one setting on the test set;
3. repeat the reference setting $p = q = 1$ with three seeds and decide whether your gain is larger than the variation between seeds.

**Predict before running:** from walkthrough W7, which $q$ do you expect to do best, and by how much?

In [8]:
rows = []
for p in [.5, 1., 2.]:
    for q in [.5, 1., 2.]:
        nodes, Zpq, _ = T.embed(G, **REFERENCE, p=p, q=q)
        rows.append({"p": p, "q": q, "validation accuracy": T.classify(Zpq, y, split)[0]["validation_accuracy"]})
grid_table = pd.DataFrame(rows)
display(grid_table.pivot(index="p", columns="q", values="validation accuracy").round(3))

q,0.5,1.0,2.0
p,,,
0.5,0.651,0.643,0.612
1.0,0.671,0.662,0.627
2.0,0.689,0.669,0.678


In [9]:
# TODO: replace None with the (p, q) you choose from the VALIDATION table, e.g. (1.0, 0.5).
chosen = None

In [10]:
if chosen is None:
    print("R2: set chosen = (p, q) from the validation table, then rerun this cell.")
else:
    nodes, Zc, _ = T.embed(G, **REFERENCE, p=chosen[0], q=chosen[1])
    result, _ = T.classify(Zc, y, split, test=True)
    print(f"setting p={chosen[0]}, q={chosen[1]}: validation {result['validation_accuracy']:.3f} | TEST {result['test_accuracy']:.3f} (macro-F1 {result['test_macro_f1']:.3f})")
    print("reference p=q=1 in W7: validation 0.662 | test 0.631")

R2: set chosen = (p, q) from the validation table, then rerun this cell.


In [11]:
seed_rows = []
for seed in [SEED, SEED + 1, SEED + 2]:
    nodes, Zs, _ = T.embed(G, **REFERENCE, p=1., q=1., seed=seed)
    seed_rows.append({"seed": seed, "validation accuracy": T.classify(Zs, y, split)[0]["validation_accuracy"]})
seeds = pd.DataFrame(seed_rows).set_index("seed")
display(seeds.round(3))
spread = seeds["validation accuracy"].max() - seeds["validation accuracy"].min()
print(f"range across three seeds for p=q=1: {spread:.3f}")

,validation accuracy
seed,
5111,0.662
5112,0.684
5113,0.649


range across three seeds for p=q=1: 0.035


**My decision (two or three sentences):** the chosen setting … its validation gain over $p = q = 1$ is … compared with a seed range of … so I conclude …

## R3 · Ranking metrics, then four edge operators
**Part 1, by hand.** Three queries have one relevant answer each, ranked 1, 2 and 5 among fixed candidates with no ties. Hits@3 is the share of queries whose answer is in the top 3; the mean reciprocal rank (MRR) averages $1/\text{rank}$. Compute both by hand first, then in code.

In [12]:
ranks = np.array([1, 2, 5])
hits_at_3 = None   # TODO: share of ranks <= 3
mrr = None         # TODO: mean of 1 / ranks

In [13]:
# Check R3, part 1
if hits_at_3 is None or mrr is None:
    print("R3: compute hits_at_3 and mrr, then rerun.")
else:
    print(f"Hits@3 = {hits_at_3:.3f} | MRR = {mrr:.3f} |", "both match" if np.isclose(hits_at_3, 2/3) and np.isclose(mrr, (1 + 1/2 + 1/5) / 3) else "check: Hits@3 = 2/3, MRR = (1 + 1/2 + 1/5)/3")

R3: compute hits_at_3 and mrr, then rerun.


Why would removing the difficult candidates from the ranking inflate both numbers? **My answer:** …

**Part 2, edge operators.** W8 combined two node vectors by the Hadamard product. Other common operators are the average $(z_u + z_v)/2$, the absolute difference $|z_u - z_v|$ (L1) and the squared difference $(z_u - z_v)^2$ (L2). Complete the two missing operators; the next cell learns one embedding on the **training graph**, trains a classifier per operator and reports validation AUC. **Predict before running:** which operator ignores how *similar* the two vectors are?

In [14]:
operators = {
    "Hadamard": lambda zu, zv: zu * zv,
    "average": lambda zu, zv: (zu + zv) / 2,
    "L1": None,   # TODO: absolute difference
    "L2": None,   # TODO: squared difference
}

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
train_graph, positive, negative = T.split_links(G)
nodes, Z_train, _ = T.embed(train_graph, **REFERENCE)          # training graph only
def operator_auc(op, part="val"):
    f = lambda P: op(Z_train[P[:, 0]], Z_train[P[:, 1]])
    Xtr = np.vstack([f(positive["train"]), f(negative["train"])]); ytr = np.r_[np.ones(len(positive["train"])), np.zeros(len(negative["train"]))]
    clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
    Xs = np.vstack([f(positive[part]), f(negative[part])]); ys = np.r_[np.ones(len(positive[part])), np.zeros(len(negative[part]))]
    return roc_auc_score(ys, clf.predict_proba(Xs)[:, 1])
done = {k: v for k, v in operators.items() if v is not None}
display(pd.Series({k: operator_auc(v) for k, v in done.items()}, name="validation AUC").round(3))
if len(done) < 4:
    print("R3: add the L1 and L2 operators, then rerun.")

Hadamard    0.772
average     0.592
Name: validation AUC, dtype: float64

R3: add the L1 and L2 operators, then rerun.


**My answer:** the best operator on validation is … ; the average performs … because …

## R4 (priority) · Two leaks in a link-prediction pipeline
The pipeline below reports a validation AUC that looks excellent. It contains **two** leaks: two places where information about the validation pairs reaches the model before it is evaluated. Run it, then read it line by line and find both. Then complete `fixed_pipeline` so that neither leak remains. The check confirms that your corrected validation AUC is close to the value of walkthrough W8 (0.772), and that the walks and the classifier saw only training edges and training pairs.

One leak inflates the score enormously; the other only a little here, because a linear classifier on 16 coordinates has little capacity to memorise pairs. A leak that changes the score little is still a leak: with a more flexible classifier the same mistake can matter a great deal.

In [16]:
def flawed_pipeline():
    nodes, z, _ = T.embed(G, **REFERENCE)
    f = lambda P: T.edge_features(z, P)
    Xtr = np.vstack([f(positive["train"]), f(positive["val"]), f(negative["train"]), f(negative["val"])])
    ytr = np.r_[np.ones(len(positive["train"]) + len(positive["val"])), np.zeros(len(negative["train"]) + len(negative["val"]))]
    clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
    Xv = np.vstack([f(positive["val"]), f(negative["val"])]); yv = np.r_[np.ones(len(positive["val"])), np.zeros(len(negative["val"]))]
    return roc_auc_score(yv, clf.predict_proba(Xv)[:, 1])

print(f"flawed pipeline, validation AUC: {flawed_pipeline():.3f}")

flawed pipeline, validation AUC: 0.979


**Leak 1:** … **Leak 2:** … (Write each in one sentence, saying why it inflates the score.)

In [17]:
def fixed_pipeline():
    graph_for_walks = None    # TODO: which graph may the walks see?
    if graph_for_walks is None:
        return None
    nodes, z, _ = T.embed(graph_for_walks, **REFERENCE)
    f = lambda P: T.edge_features(z, P)
    # TODO: build the training data from training pairs only.
    Xtr, ytr = None, None
    if Xtr is None:
        return None
    clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
    Xv = np.vstack([f(positive["val"]), f(negative["val"])]); yv = np.r_[np.ones(len(positive["val"])), np.zeros(len(negative["val"]))]
    return roc_auc_score(yv, clf.predict_proba(Xv)[:, 1]), len(ytr), graph_for_walks.number_of_edges()

In [18]:
# Check R4
result = fixed_pipeline()
if result is None:
    print("R4: complete fixed_pipeline, then rerun.")
else:
    auc, rows, edges = result
    print(f"corrected validation AUC: {auc:.3f} |", "close to W8 (0.772)" if abs(auc - .772) < .02 else "not yet close to W8 (0.772)")
    print(f"edges seen by the walks: {edges} |", "training graph only" if edges == train_graph.number_of_edges() else "the walks still see held-out edges")
    n_train = len(positive["train"]) + len(negative["train"])
    print(f"pairs used to train the classifier: {rows} |", "training pairs only" if rows == n_train else f"expected {n_train}: are validation pairs still included?")

R4: complete fixed_pipeline, then rerun.


## R5 · A two-hop neighbour average for new papers
In W9 the test papers played papers published later, and adding the mean of their earlier neighbours' words raised accuracy from 0.649 to 0.733. Here you add a second hop: the mean, over a paper's earlier neighbours, of **their** neighbour means. `T.neighbour_mean_features(G, F, visible)` returns `[F | mean of F over visible neighbours]`, so applying it to the one-hop block gives the two-hop block. Compare three representations on **validation**, choose one, then test it once.

Note one limitation of this protocol: the validation papers are earlier papers, so validation measures the representation on papers whose neighbours are all visible, and only the test set imitates new papers.

In [19]:
from scipy import sparse
new = np.zeros(len(G), bool); new[split["test"]] = True
X1, earlier = T.neighbour_mean_features(G, X, visible=~new)
one_hop = X1[:, X.shape[1]:]                     # the mean block only
print("new papers without an earlier neighbour:", int((earlier[new] == 0).sum()), "of", int(new.sum()))

new papers without an earlier neighbour: 382 of 1626


In [20]:
# TODO: build X2 = [own words | one-hop mean | two-hop mean]. Hint: apply T.neighbour_mean_features to one_hop
# with the same `visible`, keep its second block, and join the blocks with sparse.hstack([...]).tocsr().
X2 = None

In [21]:
if X2 is None:
    print("R5: build X2, then rerun.")
else:
    reps = {"own words": X, "+ one hop": X1, "+ one and two hops": X2}
    table5 = pd.DataFrame([{"features": k, "validation accuracy": T.classify(F, y, split)[0]["validation_accuracy"]} for k, F in reps.items()]).set_index("features")
    display(table5.round(3))
    choice = table5["validation accuracy"].idxmax()
    result, _ = T.classify(reps[choice], y, split, test=True)
    print(f"chosen on validation: {choice} | TEST accuracy {result['test_accuracy']:.3f} (macro-F1 {result['test_macro_f1']:.3f})")

R5: build X2, then rerun.


**My interpretation:** the second hop … ; for the new papers with no earlier neighbour, both averages are zero, so … ; a learned aggregator (GraphSAGE) could …

## R6 (optional) · TransE and a one-to-many relation
Walkthrough W10 trained TransE on six capital_of triples. Here we add a second relation, has_city, with two cities per country for Italy and France. TransE asks Italy + $r_{has\_city}$ to land on **both** Rome and Milan. **Predict before running:** what happens to the distance between Rome and Milan?

In [22]:
capitals = [("Malta", "Valletta"), ("Italy", "Rome"), ("France", "Paris"), ("Spain", "Madrid"), ("Greece", "Athens"), ("Portugal", "Lisbon")]
extra = ["Milan", "Lyon"]
entities = [c for c, _ in capitals] + [city for _, city in capitals] + extra
ix = {e: i for i, e in enumerate(entities)}
triples = [(ix[city], 0, ix[country]) for country, city in capitals]                       # relation 0: capital_of
triples += [(ix["Italy"], 1, ix["Rome"]), (ix["Italy"], 1, ix["Milan"]),
            (ix["France"], 1, ix["Paris"]), (ix["France"], 1, ix["Lyon"])]              # relation 1: has_city
E, R = T.fit_transe(triples, len(entities), 2, dim=2, epochs=800, lr=.01, margin=.5)
d = lambda a, b: float(np.linalg.norm(E[ix[a]] - E[ix[b]]))
cities = [city for _, city in capitals] + extra
typical = np.mean([d(a, b) for i, a in enumerate(cities) for b in cities[i + 1:]])
print(f"distance Rome-Milan: {d('Rome', 'Milan'):.3f} | Paris-Lyon: {d('Paris', 'Lyon'):.3f} | average between cities: {typical:.3f}")

distance Rome-Milan: 0.070 | Paris-Lyon: 0.145 | average between cities: 1.013


**My interpretation:** … (Later models such as TransH and TransR give each relation its own projection to avoid this; Session 04 develops knowledge graphs.)

## R7 · Reading note
Read only what addresses a difficulty, and answer one question for each text in a sentence or two.

- [DeepWalk: Online Learning of Social Representations](https://arxiv.org/abs/1403.6652) (Perozzi, Al-Rfou and Skiena, 2014), section 4. **Question:** which part of the algorithm did our helper replace, and with what? **Answer:** …
- [node2vec: Scalable Feature Learning for Networks](https://arxiv.org/abs/1607.00653) (Grover and Leskovec, 2016), sections 3.1–3.2. **Question:** why does the paper describe BFS-like walks as capturing structural equivalence, and what did W6 find? **Answer:** …
- Hamilton, [Graph Representation Learning](https://www.cs.mcgill.ca/~wlh/grl_book/files/GRL_Book.pdf) (2020), sections 3.1, 3.3 and 3.4. **Question:** in the encoder–decoder view, what is the encoder of DeepWalk and what is its decoder? **Answer:** …

**Feedback.** Which example helped most? Which step remains unclear? Were you able to run the cells, did you use the saved outputs, or neither? Keep these answers with the evidence listed on the Practice and Further Reading sheet.